# Caderno 02: Staging, Pré-processamento e Data Lineage

**Objetivo:** Preservar a base bruta em um banco de dados relacional e criar a tabela analítica normalizada.

**Padrão Empírico ACM SIGSOFT (Data Science):** * *Explains how the data was pre-processed, filtered, and categorized* (Explica como os dados foram pré-processados, filtrados e categorizados).

**Decisão Arquitetural (Data Lineage):**
Para mitigar ameaças à validade relacionadas à manipulação indevida dos dados, implementou-se uma arquitetura de banco de dados SQLite em duas camadas:
1. **Camada Staging (`stg_ceap`):** Ingestão do dado exatamente como fornecido pelo Senado, garantindo a imutabilidade do registro original.
2. **Camada Fato (`fato_despesa`):** Aplicação de engenharia de formatação (tipagem, limpeza de strings e adequação monetária), documentada no módulo `src/preprocess.py`.

In [5]:
import pandas as pd
import sqlite3
import glob
import sys
from pathlib import Path

# Adiciona a pasta src ao path do sistema para importar os módulos
ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

# Importa a função de limpeza
from src.preprocess import preprocessar_dataframe

# Configuração de Caminhos
RAW_DIR = ROOT_DIR / 'data' / 'raw'
DB_PATH = ROOT_DIR / 'database' / 'ceap.db'
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'

# Garante que as pastas existem
RAW_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Estabelecendo conexão com SQLite.")
conn = sqlite3.connect(DB_PATH)

Estabelecendo conexão com SQLite.


### 2.1 Carga RAW (Staging Area)
Todos os arquivos CSV da pasta `data/raw` são lidos como *strings* (texto puro). Adicionamos a coluna `source_file` para manter a rastreabilidade da linha até o arquivo original.

In [6]:
# Busca todos os arquivos coletados
arquivos_csv = glob.glob(str(RAW_DIR / 'ceap_*.csv'))
lista_dfs = []

print("Agora, iniciando a leitura dos CSVs brutos...")
for arquivo in arquivos_csv:
    # Parâmetros específicos para o CSV do Senado Federal
    df_ano = pd.read_csv(
        arquivo, 
        sep=';', 
        encoding='utf-8', 
        dtype=str, 
        on_bad_lines='skip'
    )
    # Rastreabilidade
    df_ano['source_file'] = Path(arquivo).name
    lista_dfs.append(df_ano)

# Empilha os anos e salva no banco de dados
df_raw_completo = pd.concat(lista_dfs, ignore_index=True)
df_raw_completo.to_sql('stg_ceap', conn, if_exists='replace', index=False)

print(f"Carga RAW concluída com sucesso!")
print(f"  -> Total de registros na tabela 'stg_ceap': {len(df_raw_completo):,}")

Agora, iniciando a leitura dos CSVs brutos...
Carga RAW concluída com sucesso!
  -> Total de registros na tabela 'stg_ceap': 88,180


### 2.2 Pré-processamento e Tipagem
Aplicação do pipeline de sanitização (`preprocessar_dataframe`). As seguintes operações metodológicas são realizadas:
1. **Conversão Monetária:** Substituição do padrão brasileiro (`2.102,89`) para float computacional (`2102.89`).
2. **Sanitização de Documentos:** Remoção de máscaras de CNPJ/CPF via Expressão Regular.
3. **Datas:** Conversão da string (ISO) para o tipo `datetime` nativo do pandas.
4. **Padronização Categórica:** Transformação para caixa alta e remoção de espaços nas extremidades (*strip*) para evitar fragmentação de entidades em agrupamentos futuros.

In [7]:
print("Aplicando regras de sanitização (src/preprocess.py):")

# Chama a função principal de limpeza
df_processado = preprocessar_dataframe(df_raw_completo)

# Seleção das colunas de interesse para a tabela Fato
# Note que estamos usando os nomes limpos gerados pela função preprocessar_dataframe
colunas_fato = [
    'ano', 
    'mes', 
    'cod_senador', 
    'nome_senador_limpo', 
    'tipo_despesa_limpo', 
    'nome_fornecedor_limpo', 
    'documento_fornecedor', 
    'data_despesa', 
    'valor_limpo', 
    'tipo_documento_limpo', 
    'detalhamento_limpo', 
    'source_file'
]

# Filtra o DataFrame apenas com as colunas definidas
df_fato = df_processado[colunas_fato].copy()

# Persiste a tabela limpa no banco de dados
df_fato.to_sql('fato_despesa', conn, if_exists='replace', index=False)

# Exporta fisicamente para a pasta data/processed
caminho_csv_processado = PROCESSED_DIR / 'ceap_56_legislatura_limpo.csv'
df_fato.to_csv(caminho_csv_processado, index=False, encoding='utf-8', sep=';')

print(f"Normalização concluída com sucesso!")
print(f"  -> Total de registros na tabela 'fato_despesa': {len(df_fato):,}")

# Fechando a conexão
conn.close()
print("\nPipeline de ingestão finalizado e banco de dados fechado.")

Aplicando regras de sanitização (src/preprocess.py):
Normalização concluída com sucesso!
  -> Total de registros na tabela 'fato_despesa': 88,180

Pipeline de ingestão finalizado e banco de dados fechado.
